# Maia Lite 0.3.0 — Auditoria executável do SFT v2.2 → v2.3

**Somente leitura dos dados e checkpoints originais. Não treina.** Executa verificações de integridade, cobertura, tokenização BPE, máscara causal, separadores e metadados de retomada. Produz relatórios CSV/JSON no Google Drive em uma pasta nova. Pode ser executado após reiniciar o Colab, **sem GPU**.

**Nota:** esta auditoria não demonstra, por si só, qualidade factual, legalidade das licenças ou retomada bit a bit; esses aspectos exigem avaliações adicionais.

In [1]:
# 1 — Ambiente independente
import sys, subprocess, json, hashlib, collections, random, math, re, time
from pathlib import Path
try:
    from transformers import AutoTokenizer
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers'])
    from transformers import AutoTokenizer
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
PREP=ROOT/'datasets/sft_v2_2_aya'
TRAIN_FILE=PREP/'sft_v2_2_train.jsonl'
VAL_FILE=PREP/'sft_v2_2_validation.jsonl'
AUDIT_PASS=PREP/'audit_v2_2/audit_pass.json'
TOKENIZER_DIR=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
BASE_CKPT=ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
ADAPTER=ROOT/'training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1/adapter_best.pt'
OUT=ROOT/'audits/sft_v2_2_pipeline_executable_v1'
for path in (TRAIN_FILE,VAL_FILE,AUDIT_PASS,TOKENIZER_DIR):
    if not path.exists():
        raise FileNotFoundError(f'Necessário: {path}. Confirme a montagem do Drive e o caminho.')
print('Dataset:',PREP)
print('Tokenizer:',TOKENIZER_DIR)
print('Saída (apenas relatórios):',OUT)

Mounted at /content/drive
Dataset: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/datasets/sft_v2_2_aya
Tokenizer: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer
Saída (apenas relatórios): /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/audits/sft_v2_2_pipeline_executable_v1


In [2]:
# 2 — Leitura e rastreabilidade
import pandas as pd

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def load_jsonl(path):
    with open(path,encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
train=load_jsonl(TRAIN_FILE); validation=load_jsonl(VAL_FILE)
pass_record=json.loads(AUDIT_PASS.read_text(encoding='utf-8'))
print('Train:',len(train),'Validation:',len(validation))
print('Audit pass keys:',list(pass_record))
print('SHA train:',sha256(TRAIN_FILE))
print('SHA validation:',sha256(VAL_FILE))
print('SHA audit pass:',sha256(AUDIT_PASS))
assert len(train)>0 and len(validation)>0
assert all(isinstance(r.get('prompt'),str) and isinstance(r.get('completion'),str) for r in train+validation)
assert not ({r.get('groupId') for r in train if r.get('groupId')} & {r.get('groupId') for r in validation if r.get('groupId')})
assert not ({r.get('paperId') for r in train if r.get('paperId')} & {r.get('paperId') for r in validation if r.get('paperId')})
print('Sem sobreposição de groupId/paperId não vazios.')

Train: 29995 Validation: 1521
Audit pass keys: ['created_utc', 'train_count', 'validation_count', 'train_by_language', 'train_by_source', 'issues_count', 'train_sha256', 'validation_sha256', 'human_review_approved', 'license_review_approved']
SHA train: a3db42463329f32cd56059631b8dfe29da1bc62cb034b919284c77c7603ee37b
SHA validation: 25c898dbba576914fdc68b9c83f56007bcce98a1342638495212ab1682000504
SHA audit pass: b8abe478e67dc3b8245b1ed01aeea028b4a0084585468cff11ca907507eae563
Sem sobreposição de groupId/paperId não vazios.


In [3]:
# 3 — Tokenizer original e encoder SFT v2.2 (reprodução da implementação)
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert tokenizer.is_fast, 'Necessário tokenizer fast para offsets.'
assert tokenizer.eos_token_id is not None
MAX_LENGTH=512

def encode_v22(rec, max_length=MAX_LENGTH):
    prompt=rec['prompt']; answer=rec['completion']
    if not prompt.strip() or not answer.strip():return None,'empty',{}
    enc=tokenizer(prompt+answer,add_special_tokens=False,return_offsets_mapping=True)
    ids=enc['input_ids']; offsets=enc['offset_mapping']; boundary=len(prompt)
    detail={'prompt_tokens':len(tokenizer(prompt,add_special_tokens=False)['input_ids']),
            'answer_chars':len(answer),'full_tokens_eos':len(ids)+1}
    if any(a<boundary<b for a,b in offsets):return None,'boundary_merge',detail
    answer_idx=next((i for i,(a,b) in enumerate(offsets) if a>=boundary and b>a),None)
    if answer_idx is None:return None,'no_answer_tokens',detail
    ids=ids+[tokenizer.eos_token_id]
    if len(ids)>max_length:return None,'too_long',detail
    labels=[-100]*(len(ids)-1)
    for i in range(len(labels)):
        if i+1>=answer_idx:labels[i]=ids[i+1]
    if all(v==-100 for v in labels):return None,'empty_targets',detail
    detail.update(answer_idx=answer_idx,supervised_tokens=sum(y!=-100 for y in labels))
    return {'x':ids[:-1],'y':labels,'answer_idx':answer_idx},None,detail

assert len(tokenizer)==50257, f'Vocabulário inesperado: {len(tokenizer)}'
print('Tokenizer OK:',len(tokenizer),'EOS:',tokenizer.eos_token_id)

Tokenizer OK: 50257 EOS: 2


In [4]:
# 4 — Cobertura completa, estratificada por idioma e fonte
rows=[]; accepted=[]
for split,data in [('train',train),('validation',validation)]:
    for r in data:
        encoded,reason,detail=encode_v22(r)
        source=str(r.get('source') or r.get('dataset') or r.get('origin') or 'unknown')
        rows.append({'split':split,'id':r.get('id'),'language':r.get('language'),
                     'source':source,'status':reason or 'accepted',**detail})
        if encoded is not None and len(accepted)<100:
            accepted.append((r,encoded))
coverage=pd.DataFrame(rows)
summary=coverage.groupby(['split','language','source','status'],dropna=False).size().reset_index(name='count')
print('Por motivo:')
display(coverage.groupby(['split','status']).size().unstack(fill_value=0))
print('Por idioma:')
display(coverage.groupby(['split','language','status']).size().unstack(fill_value=0))
print('Por fonte (amostra):')
display(summary.head(30))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1204 > 1024). Running this sequence through the model will result in indexing errors


Por motivo:


status,accepted,boundary_merge,too_long
split,,,
train,28857,392,746
validation,1469,20,32


Por idioma:


status               accepted  boundary_merge  too_long
split      language                                    
train      en           14540              71       388
           es            7554             123       321
           pt            6763             198        37
validation en            1017              15        17
           es             352               3        15
           pt             100               2         0

Por fonte (amostra):


,split,language,source,status,count
0,train,en,OpenAssistant/oasst1,accepted,5903
1,train,en,OpenAssistant/oasst1,boundary_merge,64
2,train,en,OpenAssistant/oasst1,too_long,388
3,train,en,allenai/ai2_arc,accepted,2371
4,train,en,allenai/ai2_arc,boundary_merge,6
5,train,en,maia_scientific_curated,accepted,812
6,train,en,openai/gsm8k,accepted,5454
7,train,en,openai/gsm8k,boundary_merge,1
8,train,es,CohereLabs/aya_dataset,accepted,1749
9,train,es,CohereLabs/aya_dataset,boundary_merge,83


In [5]:
# 5 — Propriedades da máscara causal: EOS, fronteira, rótulos e shift
rng=random.Random(20261118)
checks=[]
for split,data in [('train',train),('validation',validation)]:
    sample=rng.sample(data,min(1000,len(data)))
    for r in sample:
        item,reason,details=encode_v22(r)
        if item is None:continue
        x,y=item['x'],item['y'];idx=item['answer_idx']
        ok=(len(x)==len(y) and y[-1]==tokenizer.eos_token_id and
            all(y[i]==-100 for i in range(max(0,idx-1))) and
            all(y[i]==x[i+1] for i in range(idx-1,len(x)-1)) and
            y[idx-1]==x[idx])
        checks.append({'split':split,'id':r.get('id'),'ok':bool(ok),'answer_idx':idx,'supervised':sum(t!=-100 for t in y)})
assert checks and all(c['ok'] for c in checks), 'Falha no alinhamento causal!'
print('Máscara causal validada em',len(checks),'registros aceitos amostrados.')
print('Nota: esse teste reproduz o algoritmo do notebook; não executa o forward do modelo.')

Máscara causal validada em 1938 registros aceitos amostrados.
Nota: esse teste reproduz o algoritmo do notebook; não executa o forward do modelo.


In [6]:
# 6 — Experimento de fronteira e limites SEM modificar os dados originais
# Não é proposta definitiva de formato: mede apenas tokenização, merges e comprimento.
SEPARATORS=['', '\n', '\nResposta:\n', '\nAnswer:\n']
MAX_LENGTHS=[512,768,1024]
experiment=[]
for sep in SEPARATORS:
    for limit in MAX_LENGTHS:
        counts=collections.Counter()
        for rec in train+validation:
            variant=dict(rec);variant['prompt']=rec['prompt']+sep
            _,reason,_=encode_v22(variant,limit)
            counts[reason or 'accepted']+=1
        experiment.append({'separator':repr(sep),'max_length':limit,**counts})
experiment_df=pd.DataFrame(experiment).fillna(0)
display(experiment_df)
print('AVISO: adicionar separador muda a distribuição do treino e exige novo adaptador.')

,separator,max_length,accepted,too_long,boundary_merge
0,'',512,30326,778,412.0
1,'',768,30987,117,412.0
2,'',1024,31087,17,412.0
3,'\n',512,30722,794,0.0
4,'\n',768,31397,119,0.0
5,'\n',1024,31497,19,0.0
6,'\nResposta:\n',512,30701,815,0.0
7,'\nResposta:\n',768,31394,122,0.0
8,'\nResposta:\n',1024,31497,19,0.0
9,'\nAnswer:\n',512,30701,815,0.0


AVISO: adicionar separador muda a distribuição do treino e exige novo adaptador.


In [7]:
# 7 — Inspeção opcional dos metadados dos checkpoints (sem carregar pesos na GPU)
# torch.load(weights_only=False) só deve ser usado com arquivos próprios e confiáveis.
import torch
metadata={}
for label,path in [('base',BASE_CKPT),('adapter',ADAPTER)]:
    if not path.is_file():
        metadata[label]={'exists':False,'path':str(path)};continue
    metadata[label]={'exists':True,'path':str(path),'size_bytes':path.stat().st_size}
    if label=='adapter':
        obj=torch.load(path,map_location='cpu',weights_only=False)
        metadata[label]['keys']=list(obj)
        metadata[label]['update']=obj.get('update')
        metadata[label]['val_loss']=obj.get('val_loss')
        metadata[label]['config']=obj.get('config')
        del obj
print(json.dumps(metadata,indent=2,ensure_ascii=False,default=str)[:7000])
print('Retomada bit a bit: NÃO testada; requer execução controlada com interrupção.')

{
  "base": {
    "exists": true,
    "path": "/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt",
    "size_bytes": 6084855785
  },
  "adapter": {
    "exists": true,
    "path": "/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1/adapter_best.pt",
    "size_bytes": 14200613,
    "keys": [
      "adapter",
      "update",
      "val_loss",
      "config"
    ],
    "update": 1800,
    "val_loss": 2.319600362225873,
    "config": {
      "rank": 16,
      "alpha": 32,
      "dropout": 0.05,
      "lr": 8e-05,
      "max_length": 512,
      "micro_batch": 1,
      "grad_accum": 16,
      "epochs": 1,
      "max_steps": null,
      "seed": 20261118,
      "data_hashes": {
        "train": "a3db42463329f32cd56059631b8dfe29da1bc62cb034b919284c77c7603ee37b",
        "validation": "25c898dbba576914fdc68b9c83f56007bcce98a1342638495212ab1682000504"
      },
      "checkpoint_pat

In [8]:
# 8 — Exportar relatórios e veredito auditável
OUT.mkdir(parents=True,exist_ok=True)
coverage.to_csv(OUT/'coverage_per_record.csv',index=False)
summary.to_csv(OUT/'coverage_by_language_source.csv',index=False)
experiment_df.to_csv(OUT/'boundary_length_experiments.csv',index=False)
pd.DataFrame(checks).to_csv(OUT/'causal_mask_checks.csv',index=False)
report={
 'status':'STATIC_AND_DATA_CHECKS_COMPLETE',
 'training_executed':False,
 'bitwise_resume_tested':False,
 'human_factual_review_done':False,
 'dataset_counts':{'train':len(train),'validation':len(validation)},
 'files':{'train_sha256':sha256(TRAIN_FILE),'validation_sha256':sha256(VAL_FILE),'audit_pass_sha256':sha256(AUDIT_PASS)},
 'causal_checks':{'tested':len(checks),'failed':sum(not c['ok'] for c in checks)},
 'coverage':coverage.groupby(['split','status']).size().rename_axis(['split','status']).reset_index(name='count').to_dict('records'),
 'checkpoint_metadata':metadata,
 'recommendations':['Não alterar o mascaramento causal existente',
                    'Versionar formato de prompt/resposta antes de novo treino',
                    'Avaliar impacto de MAX_LENGTH=768/1024 em VRAM',
                    'Implementar teste real de retomada determinística',
                    'Conservar benchmark e avaliação humana independentes']
}
(OUT/'audit_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
print('AUDITORIA EXECUTADA — SEM TREINAMENTO')
for p in sorted(OUT.iterdir()):print(p)
print('Resumo:',report['coverage'])

AUDITORIA EXECUTADA — SEM TREINAMENTO
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/audits/sft_v2_2_pipeline_executable_v1/audit_report.json
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/audits/sft_v2_2_pipeline_executable_v1/boundary_length_experiments.csv
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/audits/sft_v2_2_pipeline_executable_v1/causal_mask_checks.csv
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/audits/sft_v2_2_pipeline_executable_v1/coverage_by_language_source.csv
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/audits/sft_v2_2_pipeline_executable_v1/coverage_per_record.csv
Resumo: [{'split': 'train', 'status': 'accepted', 'count': 28857}, {'split': 'train', 'status': 'boundary_merge', 'count': 392}, {'split': 'train', 'status': 'too_long', 'count': 746}, {'split': 'validation', 'status': 'accepted', 'count': 1469}, {'split': 'validation', 'status': 'boundary_merge', 'count': 20}, {'split': 'validation', 'status': 'too_long', 'cou

## Interpretação e próximos passos

- `coverage_by_language_source.csv`: mede exclusões e possíveis vieses por idioma e fonte.
- `boundary_length_experiments.csv`: compara **hipóteses** de separador e contexto; não modifica o corpus.
- `causal_mask_checks.csv`: verifica shift causal e supervisão do EOS.
- `audit_report.json`: contém hashes, resultados e limitações.

**Critério para seguir ao SFT v2.3:** revisar perdas por idioma/fonte; escolher um formato de instrução/resposta único; confirmar que o novo encoder preserva a resposta completa; implementar teste de retomada controlada; preservar os resultados do v2.2 como baseline. Não reutilize o adaptador v2.2 como se tivesse sido treinado no novo formato.